# Business Entity Resolution — LOCAL Implementation Notebook (Windows)

**Runs on:** local Windows PC (Ultra 9 285H, 32 GB RAM, Intel Arc 140T iGPU) with OpenVINO for transformer inference.
**Sibling:** `entity_resolution.ipynb` is the SageMaker `ml.t3.medium` (2 vCPU / 4 GB) variant — untouched. This file is the local counterpart: bigger sample budgets, Windows-safe memory reporting, OpenVINO install + device check, and a full-scale blocking recipe sized for 32 GB instead of a t3.medium refusal stub.
**Plan ref:** `../ENTITY_RESOLUTION_PLAN.md` — Phase 0 → 0.5 → 1 → 2 → 3/4/5. All `.tsv` reads use `sep="\t"`.

Expected data layout (repo root on disk, already the case):
```
<DATA_ROOT>/train/train_source1.tsv
<DATA_ROOT>/train/train_source2.tsv
<DATA_ROOT>/train/train_source3.tsv
<DATA_ROOT>/train/train_ground_truth.tsv
<DATA_ROOT>/test/test_source1.tsv ...
```

## Which cells to run / skip (read this first)

**First run:** Run All, but expect exactly ONE red cell - the `RUN_FULL` assert (needs ~20 GB free + the full Phase-2 recipe implemented). A red `RUN_FULL` cell means the guardrail works.

| Cell | Verdict |
|---|---|
| `%pip install` | Run once - skip on re-runs unless the venv was recreated |
| Config (RAM budgets), scorer + unit test, val split, normalization lib | Must run every time |
| Skeleton run | Must run - the core pipeline |
| TSV write + validation | Must run - must print `PASS` |
| Drive detect | Must run for sharing - prints local-only when Drive is not mounted (set GDRIVE if letter differs) |
| Sync stage (push/pull) | Safe to always run - plain file copies via Drive for Desktop, resumes via manifest |
| France asserts | Optional - instant sanity check |
| FAISS ANN demo | Optional - smoke test at 100k-doc scale |
| OpenVINO device check | Must run (cheap) - confirms Arc `GPU` is visible |
| OV inference stub | Safe to always run - self-skips with a message until `models/minilm-ov` exists |
| Phase 5 smoke | Run first (~10 min) - must print SMOKE GATE: PASS before any full launch |
| Phase 5 full test | `TEST_SHARDS` per device (`["India"]` / `["US","France"]`), `RUN_TEST=True` (~26h total, resumable per chunk) |
| Phase 5 merge | Merge machine: gather all `test_parts_full/*.tsv`, run concat + coverage gate |
| Summary print | Optional |


In [36]:
%pip install -q faiss-cpu rapidfuzz lightgbm openvino optimum-intel psutil sparse-dot-topn tqdm


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import hashlib
import os
import platform
import re
import time
import unicodedata
from pathlib import Path

# ---- HARDWARE CONFIGURATION: PIN TO 8 cores ONLY ----
# Core Ultra 9 285H has 8 Performance cores, 6 Efficient cores, and 2 LP-E cores.
# Restrict all compute libraries strictly to 8 threads to run only on high-speed cores,
# avoiding straggler latency on E-cores and leaving them free for Windows OS responsiveness.
NUM_P_CORES = 12
os.environ["OMP_NUM_THREADS"] = str(NUM_P_CORES)
os.environ["MKL_NUM_THREADS"] = str(NUM_P_CORES)
os.environ["OPENBLAS_NUM_THREADS"] = str(NUM_P_CORES)
os.environ["NUMEXPR_NUM_THREADS"] = str(NUM_P_CORES)

import numpy as np
import pandas as pd
import psutil

PROC = psutil.Process()

def rss_gb() -> float:
    return PROC.memory_info().rss / 1e9

try:
    from tqdm.auto import tqdm as _tqdm
except Exception:
    _tqdm = None  # fallback: plain loops, stage prints still show progress


def pbar(it, total=None, desc=""):
    """tqdm bar when available, else the bare iterator (with one warning)."""
    if _tqdm is not None:
        return _tqdm(it, total=total, desc=desc, miniters=1)
    if not getattr(pbar, "warned", False):
        pbar.warned = True
        print("pbar: tqdm missing \u2014 plain loops (rerun the %pip cell, then restart kernel, for bars)", flush=True)
    return it

TOTAL_GB = psutil.virtual_memory().total / 1e9
print(f"OS={platform.system()} CPUs={os.cpu_count()} (Using {NUM_P_CORES} cores) RAM={TOTAL_GB:.1f} GB")

# ---- RAM-based budgets (32 GB local vs 4 GB t3.medium fallback) ----
if TOTAL_GB >= 16:
    SAMPLE_S1, POOL_DOCS = 20000, 100000
    MODE = "local-32GB"
else:
    SAMPLE_S1, POOL_DOCS = 5000, 15000
    MODE = "low-ram-fallback"

TOP_K = 10
RANDOM_STATE = 42
RUN_FULL = False  # Set True to block all 224k val S1 queries; False runs fast on SAMPLE_S1 (20k)
MODEL_DIR = Path("models/minilm-ov")
OUT_DIR = Path("output-local"); OUT_DIR.mkdir(exist_ok=True)

# ---- Drive share via Drive for Desktop ----
DRIVE_PARENT = "AmazonMLChallenge"
DRIVE_SHARE = "shared"
MANIFEST = Path("sync_manifest.json")

def find_drive():
    cands = [os.environ.get("GDRIVE"), "G:/My Drive", "H:/My Drive",
             "I:/My Drive", "F:/My Drive", "E:/My Drive"]
    for c in cands:
        if c and Path(c).is_dir():
            return Path(c)
    return None

print(f"MODE={MODE} SAMPLE_S1={SAMPLE_S1} POOL_DOCS={POOL_DOCS}")

# ---- data root autodetect ----
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent, Path.home(),
                   Path.home() / "Amazon_ML_Challange"]
DATA_ROOT = None
for r in CANDIDATE_ROOTS:
    for c in [r / "dataset",
              r / "student_resource" / "student_resource" / "dataset",
              r / "student_resource" / "dataset"]:
        if (c / "train" / "train_source1.tsv").exists():
            DATA_ROOT = c
            break
    if DATA_ROOT is not None:
        break
print("DATA_ROOT =", DATA_ROOT)
assert DATA_ROOT is not None, "Dataset not found - run from repo checkout."



OS=Windows CPUs=16 (Using 12 cores) RAM=33.7 GB
MODE=local-32GB SAMPLE_S1=20000 POOL_DOCS=100000
DATA_ROOT = d:\Amazon_ML_Challange\student_resource\student_resource\dataset


c:\Users\Mitansh\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---- Drive-for-Desktop detect (None = offline; pipeline unaffected) ----
# No OAuth, no tokens: if Drive for Desktop is running (usually G:\My Drive),
# the share folder is just another local path. Set the GDRIVE env var if your
# drive letter differs. Re-run this cell after (re)connecting Drive.
GDRIVE = find_drive()
if GDRIVE is None:
    print("drive not mounted - local-only mode (outputs stay in output-local/).")
    SHARE_ROOT = None
else:
    SHARE_ROOT = GDRIVE / DRIVE_PARENT / DRIVE_SHARE
    print("drive path:", GDRIVE, "| share:", SHARE_ROOT)


drive not mounted - local-only mode (outputs stay in output-local/).


## Phase 0 — Macro-F0.5 scorer (with unit test)
Singleton semantics: empty/empty → 1.0; any false merge on a singleton → 0.0.

In [3]:
def f05_single(y_true: set, y_pred: set) -> float:
    tp = len(y_true & y_pred)
    prec = tp / len(y_pred) if y_pred else (1.0 if not y_true else 0.0)
    rec = tp / len(y_true) if y_true else (1.0 if not y_pred else 0.0)
    if prec + rec == 0:
        return 0.0
    return 1.25 * prec * rec / (0.25 * prec + rec)


def macro_f05(truth: dict, pred: dict) -> float:
    return sum(f05_single(set(truth[k]), set(pred.get(k, ()))) for k in truth) / len(truth)


# unit tests: PDF worked example S1-00001 -> P=2/3, R=1.0, F0.5=0.7142857
assert abs(f05_single({"S2-00047", "S3-00812"}, {"S2-00047", "S2-00193", "S3-00812"}) - 0.7142857) < 1e-6
assert f05_single(set(), set()) == 1.0
assert f05_single(set(), {"S2-1"}) == 0.0
assert f05_single({"S2-1"}, set()) == 0.0
print("scorer OK: pdf-example=0.7142857, singleton-empty=1.0, singleton-fp=0.0")

scorer OK: pdf-example=0.7142857, singleton-empty=1.0, singleton-fp=0.0


## Phase 0 — Validation split (hash-based, single chunked pass)
Deterministic 10% of S1 ids (`md5 % 10 == 0`), stratified *reporting* by match-count bucket. GT streams in chunks; only val rows kept (~220k).

In [4]:
def in_val(s1: str) -> bool:
    return hashlib.md5(s1.encode()).digest()[0] % 10 == 0


def bucket(n: int) -> str:
    if n == 0:
        return "0-singleton"
    if n == 1:
        return "1"
    if n <= 3:
        return "2-3"
    if n <= 5:
        return "4-5"
    return "6+"


val_matches: dict = {}
total = 0
t0 = time.time()
for ch in pd.read_csv(DATA_ROOT / "train" / "train_ground_truth.tsv", sep="\t", chunksize=200000):
    s1s = ch["source1_entity_id"].astype(str)
    ms = ch["matched_entity_ids"].fillna("").astype(str)
    for s1, m in zip(s1s, ms):
        total += 1
        if in_val(s1):
            m = m.strip()
            val_matches[s1] = [x for x in m.split(",") if x] if m else []
print(f"GT rows={total} val_S1={len(val_matches)} ({len(val_matches)/total:.1%}) in {time.time()-t0:.0f}s, rss={rss_gb():.2f} GB")
dist: dict = {}
for v in val_matches.values():
    b = bucket(len(v))
    dist[b] = dist.get(b, 0) + 1
print("val bucket distribution:", dist)

GT rows=2206821 val_S1=224776 (10.2%) in 6s, rss=0.29 GB
val bucket distribution: {'2-3': 92404, '4-5': 82112, '1': 11976, '6+': 25658, '0-singleton': 12626}


## Phase 0.5 — Walking skeleton (trivial PIN blocking, full pipeline)
Integration first: prove load → normalize → block → features → score → TSVs → validation PASS locally before building real blocking.

In [5]:
US_IN_ABBR = {
    "corp": "corporation", "inc": "incorporated", "pvt": "private",
    "ltd": "limited", "rd": "road", "st": "street", "ave": "avenue",
    "blvd": "boulevard", "ste": "suite", "apt": "apartment",
    "mfg": "manufacturing", "ent": "enterprises", "co": "company",
    "llc": "limited liability company", "llp": "limited liability partnership",
    "dba": "doing business as", "nagar": "nagar", "marg": "marg",
    "dr": "drive", "hwy": "highway", "fl": "floor", "bldg": "building"
}
FR_ABBR = {
    "sarl": "societe responsabilite limitee", "sas": "societe actions simplifiee",
    "sa": "societe anonyme", "eurl": "entreprise unipersonnelle",
    "sasu": "societe actions simplifiee unipersonnelle",
    "rue": "rue", "bd": "boulevard", "av": "avenue", "pl": "place",
    "imp": "impasse", "cedex": "cedex", "ste": "societe", "ets": "etablissements"
}
ABBR = {**US_IN_ABBR, **FR_ABBR}
LEGAL_SUFFIX = set(ABBR) | {"corporation", "incorporated", "private", "limited",
                "company", "llc", "llp", "gmbh", "societe"}
PIN_RE = r"(?<!\d)(\d{5,6})(?!\d)"


def normalize_text(s: str) -> str:
    """Preserves Unicode scripts (Hindi, Kannada, etc.) for bilingual matching."""
    s = unicodedata.normalize("NFKC", str(s))
    s = s.lower().replace("&", " and ")
    s = re.sub(r"^(m\s*/\s*s|m\s*\.\s*s)\.?\s+", "", s)
    s = re.sub(r"\bd\s*/\s*b\s*/\s*a\b", "dba", s)
    s = re.sub(r"\b([a-z])\.(?=[a-z]\b|\s|[a-z]\.|$)", r"\1", s)
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    toks = [ABBR.get(t, t) for t in s.split()]
    return re.sub(r"\s+", " ", " ".join(toks)).strip()


def normalize_ascii(s: str) -> str:
    """ASCII-only normalization used for TF-IDF char n-gram representation."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    s = s.lower().replace("&", " and ")
    s = re.sub(r"^(m\s*/\s*s|m\s*\.\s*s)\.?\s+", "", s)
    s = re.sub(r"\bd\s*/\s*b\s*/\s*a\b", "dba", s)
    s = re.sub(r"\b([a-z])\.(?=[a-z]\b|\s|[a-z]\.|$)", r"\1", s)
    s = re.sub(r"\.(com|org|net|co|in|biz|info)\b", " ", s)
    s = re.sub(r"\s+-\s*\d[\d\s]*$", "", s)
    s = re.sub(r"(?<=[a-z])1(?=[a-z])", "l", s)
    s = re.sub(r"(?<=[a-z])0(?=[a-z])", "o", s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    toks = [ABBR.get(t, t) for t in s.split()]
    return re.sub(r"\s+", " ", " ".join(toks)).strip()


def extract_pin(addr: str) -> str:
    m = re.search(PIN_RE, str(addr))
    return m.group(1) if m else ""


print("SARL Dupont ->", normalize_text("SARL Dupont"))
print("S.A.R.L. Dupont ->", normalize_text("S.A.R.L. Dupont"))
print("M/s Reliance Industries Pvt. Ltd. ->", normalize_text("M/s Reliance Industries Pvt. Ltd."))
assert normalize_ascii("Ho1y Ministries - 1535566039") == normalize_ascii("Holy Ministries")
assert normalize_ascii("quinlandentalassociates.com") == "quinlandentalassociates"
assert normalize_ascii("oncologyplatinumspecialists.com") == "oncologyplatinumspecialists"
print("v4 US-recall normalization asserts OK")


SARL Dupont -> societe responsabilite limitee dupont
S.A.R.L. Dupont -> societe responsabilite limitee dupont
M/s Reliance Industries Pvt. Ltd. -> reliance industries private limited
v4 US-recall normalization asserts OK


In [6]:
from rapidfuzz import fuzz

sample_s1 = sorted(val_matches)[:SAMPLE_S1]
sample_set = set(sample_s1)
print(f"skeleton queries: {len(sample_s1)}, rss={rss_gb():.2f} GB")

# 1. load sample S1 rows (chunked scan, keep targets only)
s1_parts = []
for ch in pd.read_csv(DATA_ROOT / "train" / "train_source1.tsv", sep="\t", chunksize=200000):
    hit = ch[ch["entity_id"].astype(str).isin(sample_set)]
    if len(hit):
        s1_parts.append(hit)
s1_df = pd.concat(s1_parts, ignore_index=True)
assert len(s1_df) == len(sample_s1), (len(s1_df), len(sample_s1))
s1_df["norm_name"] = s1_df["business_name"].fillna("").map(normalize_text)
s1_df["norm_addr"] = s1_df["business_address"].fillna("").map(normalize_text)
s1_df["pin"] = s1_df["business_address"].fillna("").map(extract_pin)
print(f"sample S1 with PIN: {(s1_df['pin'] != '').sum()} / {len(s1_df)}, rss={rss_gb():.2f} GB")
pin_to_s1 = set(s1_df.loc[s1_df["pin"] != "", "pin"])

# 2. PIN-blocked pool scan over S2+S3 (vectorized extract per chunk; uses all cores)
t0 = time.time()
pool_parts = []
for name in ["train_source2.tsv", "train_source3.tsv"]:
    for ch in pd.read_csv(DATA_ROOT / "train" / name, sep="\t", chunksize=200000):
        pins = ch["business_address"].fillna("").str.extract(PIN_RE, expand=False)
        hit = ch[pins.isin(pin_to_s1)]
        if len(hit):
            pool_parts.append(hit)
pool_df = pd.concat(pool_parts, ignore_index=True).drop_duplicates("entity_id") if pool_parts else s1_df.iloc[0:0].copy()
pool_df["norm_name"] = pool_df["business_name"].fillna("").map(normalize_text)
pool_df["norm_addr"] = pool_df["business_address"].fillna("").map(normalize_text)
pool_df["pin"] = pool_df["business_address"].fillna("").map(extract_pin)
print(f"pool rows sharing a PIN: {len(pool_df)} in {time.time()-t0:.0f}s, rss={rss_gb():.2f} GB")

# 3. candidates per S1 (cap 40 = sanity bound)
pool_by_pin: dict = {}
for r in pool_df.itertuples():
    pool_by_pin.setdefault(r.pin, []).append(r.entity_id)
pool_idx = {r.entity_id: r for r in pool_df.itertuples()}
s1_idx = {r.entity_id: r for r in s1_df.itertuples()}
candidates = {s: list(pool_by_pin.get(s1_idx[s].pin, []))[:40] for s in sample_s1}
print(f"mean K={float(np.mean([len(v) for v in candidates.values()])):.1f}")

# 4. score (skeleton: untrained weighted sum) + threshold -> matches
def pair_score(a, b) -> float:
    ratio = fuzz.WRatio(a.norm_name, b.norm_name) / 100.0
    ta, tb = set(a.norm_name.split()), set(b.norm_name.split())
    jac = len(ta & tb) / max(1, len(ta | tb))
    pin = 1.0 if (a.pin and a.pin == b.pin) else 0.0
    return 0.5 * ratio + 0.3 * jac + 0.2 * pin


TAU_SKELETON = 0.5
matches = {}
for s in sample_s1:
    a = s1_idx[s]
    scored = sorted(((pool_idx[c], pair_score(a, pool_idx[c])) for c in candidates[s]),
                    key=lambda t: -t[1])
    matches[s] = [r.entity_id for r, sc in scored if sc >= TAU_SKELETON]
print(f"skeleton macro-F0.5 on sample: {macro_f05({s: val_matches[s] for s in sample_s1}, matches):.4f}")
print(f"rss={rss_gb():.2f} GB")

skeleton queries: 20000, rss=0.29 GB
sample S1 with PIN: 1377 / 20000, rss=0.35 GB
pool rows sharing a PIN: 60698 in 26s, rss=0.33 GB
mean K=1.9
skeleton macro-F0.5 on sample: 0.0880
rss=0.34 GB


In [7]:
# 5. write TSVs (tab-separated, utf-8) + validate
def write_id_list(path: Path, rows: dict, col: str):
    df = pd.DataFrame({"source1_entity_id": list(rows.keys()),
                       col: [",".join(rows[k]) for k in rows]})
    df.to_csv(path, sep="\t", index=False, encoding="utf-8")


write_id_list(OUT_DIR / "matching_results.tsv", matches, "matched_entity_ids")
write_id_list(OUT_DIR / "candidate_pairs.tsv", candidates, "candidate_entity_ids")
print("wrote", OUT_DIR / "matching_results.tsv", "and", OUT_DIR / "candidate_pairs.tsv")


def check_outputs(match_path: Path, cand_path: Path, required: set):
    issues = []
    m = pd.read_csv(match_path, sep="\t", keep_default_na=False)
    c = pd.read_csv(cand_path, sep="\t", keep_default_na=False)
    if list(m.columns) != ["source1_entity_id", "matched_entity_ids"]:
        issues.append(f"matching header: {list(m.columns)}")
    if list(c.columns) != ["source1_entity_id", "candidate_entity_ids"]:
        issues.append(f"candidate header: {list(c.columns)}")
    if set(m["source1_entity_id"]) != required or len(m) != len(required):
        issues.append("matching S1 row coverage mismatch")
    if set(c["source1_entity_id"]) != required or len(c) != len(required):
        issues.append("candidate S1 row coverage mismatch")
    valid = set(pool_df["entity_id"].astype(str))
    cmap = {}
    for _, r in c.iterrows():
        ids = [x for x in str(r["candidate_entity_ids"]).split(",") if x]
        if len(ids) != len(set(ids)):
            issues.append(f"dupes in candidates for {r['source1_entity_id']}")
        bad = [x for x in ids if not x.startswith(("S2-", "S3-")) or x not in valid]
        if bad:
            issues.append(f"bad candidate ids for {r['source1_entity_id']}: {bad[:3]}")
        cmap[r["source1_entity_id"]] = set(ids)
    for _, r in m.iterrows():
        ids = [x for x in str(r["matched_entity_ids"]).split(",") if x]
        if len(ids) != len(set(ids)):
            issues.append(f"dupes in matches for {r['source1_entity_id']}")
        if not set(ids) <= cmap.get(r["source1_entity_id"], set()):
            issues.append(f"match not in candidates for {r['source1_entity_id']}")
    return issues


issues = check_outputs(OUT_DIR / "matching_results.tsv",
                         OUT_DIR / "candidate_pairs.tsv", set(sample_s1))
print("VALIDATION:", "PASS" if not issues else f"FAIL {issues[:5]}")

wrote output-local\matching_results.tsv and output-local\candidate_pairs.tsv
VALIDATION: PASS


In [8]:
# ---- Sync stage: mirror to the Drive share folder (best-effort file copies) ----
# Local files are the source of truth. sync_manifest.json records hashes so only
# changed files move; reconnecting after offline days resumes the delta. Copies go
# via temp name + os.replace (atomic), then hash-verified. Any failure degrades to
# local-only - local outputs are always intact.
import hashlib
import json as _json
import os
import shutil


def md5_of(p: Path) -> str:
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(4 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_manifest() -> dict:
    try:
        return _json.loads(MANIFEST.read_text())
    except Exception:
        return {}


def push_file(local: Path, dest_dir: Path, key: str, manifest: dict, stats: dict):
    digest = md5_of(local)
    dest = dest_dir / local.name
    if manifest.get(key) == digest and dest.exists():
        stats["skip"] += 1
        return
    dest_dir.mkdir(parents=True, exist_ok=True)
    tmp = dest_dir / (local.name + ".part")
    shutil.copy2(local, tmp)
    os.replace(tmp, dest)  # atomic on the same filesystem
    assert md5_of(dest) == digest, f"verify failed: {local.name}"
    manifest[key] = digest
    stats["push"] += 1
    print(f"  pushed {local.name}")


def push_tree(local_dir: Path, dest_dir: Path, key_prefix: str, manifest: dict, stats: dict):
    if not local_dir.is_dir():
        return
    for p in sorted(local_dir.iterdir()):
        if p.is_file():
            push_file(p, dest_dir, f"{key_prefix}/{p.name}", manifest, stats)


def pull_tree(src_dir: Path, local_dir: Path, key_prefix: str, manifest: dict, stats: dict):
    if not src_dir.is_dir():
        return  # nothing shared yet - not an error
    local_dir.mkdir(parents=True, exist_ok=True)
    for p in sorted(src_dir.iterdir()):
        if not p.is_file() or p.suffix == ".part":
            continue
        key = f"{key_prefix}/{p.name}"
        digest = md5_of(p)
        dest = local_dir / p.name
        if manifest.get(key) == digest and dest.exists() and md5_of(dest) == digest:
            stats["skip"] += 1
            continue
        tmp = local_dir / (p.name + ".part")
        shutil.copy2(p, tmp)
        os.replace(tmp, dest)
        manifest[key] = digest
        stats["pull"] += 1
        print(f"  pulled {p.name}")


manifest = load_manifest()
if SHARE_ROOT is None:
    print("OFFLINE: local outputs kept; will sync once Drive is mounted.")
else:
    try:
        t0 = time.time()
        stats = {"push": 0, "skip": 0, "pull": 0}
        drive_base = GDRIVE / DRIVE_PARENT
        for p in sorted(OUT_DIR.glob("*.tsv")):
            push_file(p, SHARE_ROOT / "outputs", f"push:shared/outputs/{p.name}", manifest, stats)
        push_tree(Path("models/minilm-ov"), SHARE_ROOT / "models", "push:shared/models", manifest, stats)
        push_tree(Path("models/minilm-er"), SHARE_ROOT / "models", "push:shared/models", manifest, stats)
        lgbm = Path("models/lgbm.txt")
        if lgbm.exists():
            push_file(lgbm, SHARE_ROOT / "models", "push:shared/models/lgbm.txt", manifest, stats)
        pull_tree(drive_base / "models" / "minilm-ov", Path("models/minilm-ov"), "pull:models/minilm-ov", manifest, stats)
        pull_tree(drive_base / "models" / "minilm-er", Path("models/minilm-er"), "pull:models/minilm-er", manifest, stats)
        pull_tree(drive_base / "checkpoints" / "minilm-er", Path("models/minilm-er"), "pull:checkpoints/minilm-er", manifest, stats)
        MANIFEST.write_text(_json.dumps(manifest, indent=1))
        print(f"SYNC done in {time.time()-t0:.0f}s: {stats}")
    except OSError as e:
        print(f"SYNC interrupted ({e}) - local files intact; resumes next run (Drive may have disconnected).")


OFFLINE: local outputs kept; will sync once Drive is mounted.


## Phase 1 — Normalization check incl. France

In [9]:
assert "societe responsabilite limitee dupont" in normalize_text("SARL Dupont")
assert "societe responsabilite limitee dupont" in normalize_text("S.A.R.L. Dupont")
assert "reliance industries private limited" in normalize_text("M/s Reliance Industries Pvt. Ltd.")
assert normalize_ascii("Lumay Boral") == normalize_ascii("Lumay Boral")
print("All normalization checks (France, India M/s, dotted acronyms, Unicode-safe) OK")


All normalization checks (France, India M/s, dotted acronyms, Unicode-safe) OK


## Phase 2 — FAISS ANN demo at local scale (primary index, per plan)
Dense `IndexFlatIP` over hashed char 3–5g vectors. At 100k docs × 4096-dim ≈ 1.6 GB — comfortable on 32 GB, impossible on t3.medium. Recall denominator = matches present in pool.

In [10]:
import faiss
from sklearn.feature_extraction.text import HashingVectorizer

try:
    faiss.omp_set_num_threads(NUM_P_CORES)
except Exception:
    pass
print("faiss threads:", faiss.omp_get_max_threads())

demo_pool = pool_df.head(POOL_DOCS).reset_index(drop=True)
pool_texts = (demo_pool["norm_name"] + " [SEP] " + demo_pool["norm_addr"]).tolist()
q_texts = [(s1_idx[s].norm_name + " [SEP] " + s1_idx[s].norm_addr) for s in sample_s1]
q_ids = [str(x) for x in demo_pool["entity_id"]]
pool_set = set(q_ids)

vec = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 5), n_features=4096,
                        alternate_sign=False, norm="l2")
t0 = time.time()
X = vec.transform(pool_texts).astype(np.float32).toarray()
index = faiss.IndexFlatIP(X.shape[1])
index.add(X)
Q = vec.transform(q_texts).astype(np.float32).toarray()
D, I = index.search(Q, TOP_K)
print(f"FAISS demo: pool={len(demo_pool)} dim={X.shape[1]} topK={TOP_K} in {time.time()-t0:.1f}s, rss={rss_gb():.2f} GB")

recalls, covered = [], 0
for i, s in enumerate(sample_s1):
    denom = [m for m in val_matches[s] if m in pool_set]
    if not denom:
        continue
    covered += 1
    hits = [q_ids[j] for j in I[i]]
    recalls.append(len(set(hits) & set(denom)) / len(denom))
print(f"pool coverage of sample GT: {covered}/{len(sample_s1)}")
if recalls:
    print(f"recall@{TOP_K} (in-pool denom) mean={np.mean(recalls):.3f}")
print("NOTE: skeleton pool is PIN-biased by construction — index smoke test, not a recall claim.")


faiss threads: 12
FAISS demo: pool=60698 dim=4096 topK=10 in 30.1s, rss=2.77 GB
pool coverage of sample GT: 1233/20000
recall@10 (in-pool denom) mean=0.984
NOTE: skeleton pool is PIN-biased by construction — index smoke test, not a recall claim.


## Phase 2 — Country-Split Blocking (India Full-Address Bridge + US Name-Dominant)
Applies specialized blocking per country (0 cross-country matches in data):
- India: name×2 + FULL address (bridges cross-script matches where names are Hindi/Kannada but addresses are Roman English)
- US (v4): name×2 + first-5 address tokens (lifts concatenated-domain matches; collisions bounded by max_df=0.35)
- Western/Default: name×2 + address (supports France in test)
Runs on 8 P-cores via sparse_dot_topn with automatic shard checkpointing.


In [11]:
import collections
import gc
import pickle
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn


def soundex(tok: str) -> str:
    """American Soundex of a single token; '' for empty. Used only for gated backfill."""
    tok = re.sub(r"[^a-z]", "", tok.lower())
    if not tok:
        return ""
    _map = {"b": "1", "f": "1", "p": "1", "v": "1", "c": "2", "g": "2",
            "j": "2", "k": "2", "q": "2", "s": "2", "x": "2", "z": "2",
            "d": "3", "t": "3", "l": "4", "m": "5", "n": "5", "r": "6"}
    digits = re.sub(r"(\d)\1+", r"\1", "".join(_map.get(ch, "") for ch in tok[1:]))
    return (tok[0].upper() + digits + "000")[:4]


print(f"=== Phase 2: Country-Split TF-IDF Blocking ===")
print(f"Hardware: Pinned strictly to {NUM_P_CORES} P-cores. RSS: {rss_gb():.2f} GB")

# 1. Select target queries
target_s1_list = sorted(val_matches.keys()) if RUN_FULL else sample_s1
target_s1_set = set(target_s1_list)
print(f"Target S1 queries: {len(target_s1_set):,} (RUN_FULL={RUN_FULL})")

# 2. Load target S1 query records
t0 = time.time()
q_records = []
for ch in pd.read_csv(DATA_ROOT / "train" / "train_source1.tsv", sep="\t", chunksize=200000):
    hit = ch[ch["entity_id"].astype(str).isin(target_s1_set)]
    if len(hit):
        q_records.append(hit)
q_df = pd.concat(q_records, ignore_index=True)
q_df["norm_name"] = q_df["business_name"].fillna("").map(normalize_text)
q_df["norm_addr"] = q_df["business_address"].fillna("").map(normalize_text)
q_df["ascii_name"] = q_df["business_name"].fillna("").map(normalize_ascii)
q_df["ascii_addr"] = q_df["business_address"].fillna("").map(normalize_ascii)
q_df["country"] = q_df["country"].fillna("").str.strip()
print(f"Loaded {len(q_df):,} query records in {time.time()-t0:.1f}s, RSS={rss_gb():.2f} GB")

# 3. Country-Specific Search Text Builders
def make_search_text_india(name, addr):
    # Repeat name twice + FULL address (the essential bridge for Hindi/Kannada cross-script entities)
    return f"{name} {name} {addr}".strip()

def make_search_text_us(name, addr):
    # v4: name x2 (lifts concatenated-domain matches via char n-grams) + first 5 address
    # tokens (extra context vs v3's 3; franchise collisions still bounded by max_df=0.35).
    addr_toks = addr.split()[:5]
    return f"{name} {name} {' '.join(addr_toks)}".strip()

def make_search_text_default(name, addr):
    return f"{name} {name} {addr}".strip()

COUNTRY_CONFIGS = {
    "India": {
        "text_fn": make_search_text_india,
        "top_k": 90,
        "min_df": 2,
        "max_df": 0.40,
    },
    "US": {
        "text_fn": make_search_text_us,
        "top_k": 80,
        "min_df": 3,
        "max_df": 0.35,
    },
    "__default__": {
        "text_fn": make_search_text_default,
        "top_k": 40,
        "min_df": 2,
        "max_df": 0.35,
    }
}

# 4. Country-Sharded Blocking Loop
CHECKPOINT_DIR = OUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
USE_BASELINE_US = False  # True = roll back to 93.17% v3 US baseline, ignore v4
BASELINE_US_CKPT = CHECKPOINT_DIR / "candidates_v3_us_11935_BASELINE_9317.pkl"
LOCKED_CKPTS = {"India": "candidates_v4_india_8065.pkl",  # frozen Phase-2 winners, 94.15% overall
                "US": "candidates_v5_us_11935.pkl"}
USE_LOCKED = True  # True = load frozen winners in seconds, no recompute; False = re-tune path

BATCH_SIZE = 2500
candidates_all = {}
scores_all = {}

unique_countries = sorted(q_df["country"].unique())
print(f"Countries to process: {unique_countries}")

for ctry in unique_countries:
    t_shard = time.time()
    cfg = COUNTRY_CONFIGS.get(ctry, COUNTRY_CONFIGS["__default__"])
    text_fn = cfg["text_fn"]
    top_k = cfg["top_k"]
    
    ctry_q_df = q_df[q_df["country"] == ctry].reset_index(drop=True)
    n_queries = len(ctry_q_df)
    if n_queries == 0:
        continue
        
    print(f"\n--- Shard: '{ctry}' ({n_queries:,} queries, top_k={top_k}) ---")
    
    # Check cache
    if ctry == "US" and USE_BASELINE_US and BASELINE_US_CKPT.exists():
        print(f"  USE_BASELINE_US=True: loading {BASELINE_US_CKPT.name}...")
        with open(BASELINE_US_CKPT, "rb") as f:
            shard_data = pickle.load(f)
            candidates_all.update(shard_data["candidates"])
            scores_all.update(shard_data["scores"])
        continue

    _ver = "v4" if ctry == "India" else "v5"  # India top-90 (v3 baseline kept on disk); US top-80
    _lname = LOCKED_CKPTS.get(ctry, "") if USE_LOCKED else ""
    if _lname:
        _locked = CHECKPOINT_DIR / _lname
        if _locked.exists():
            print(f"  LOCKED: loading {_locked.name} (frozen winner, no recompute)...")
            with open(_locked, "rb") as f:
                shard_data = pickle.load(f)
                candidates_all.update(shard_data["candidates"])
                scores_all.update(shard_data["scores"])
            continue
        print(f"  LOCKED file missing ({_lname}) — falling back to compute path")
    ckpt_path = CHECKPOINT_DIR / f"candidates_{_ver}_{ctry.lower()}_{n_queries}.pkl"
    if ckpt_path.exists():
        print(f"  Loading cached candidates from {ckpt_path.name}...")
        with open(ckpt_path, "rb") as f:
            shard_data = pickle.load(f)
            candidates_all.update(shard_data["candidates"])
            scores_all.update(shard_data["scores"])
        continue

    # Build query search texts for this shard
    ctry_q_df["search_text"] = [text_fn(n, a) for n, a in zip(ctry_q_df["ascii_name"], ctry_q_df["ascii_addr"])]
    
    # Stream S2 and S3 pool for this country
    pool_records = []
    t_load = time.time()
    for src_file in ["train_source2.tsv", "train_source3.tsv"]:
        for ch in pd.read_csv(DATA_ROOT / "train" / src_file, sep="\t", chunksize=250000):
            ch["country"] = ch["country"].fillna("").str.strip()
            hit = ch[ch["country"] == ctry]
            if len(hit):
                pool_records.append(hit[["entity_id", "business_name", "business_address"]])

    pool_df_ctry = pd.concat(pool_records, ignore_index=True).drop_duplicates("entity_id")
    del pool_records
    gc.collect()

    pool_df_ctry["ascii_name"] = pool_df_ctry["business_name"].fillna("").map(normalize_ascii)
    pool_df_ctry["ascii_addr"] = pool_df_ctry["business_address"].fillna("").map(normalize_ascii)
    pool_df_ctry["search_text"] = [text_fn(n, a) for n, a in zip(pool_df_ctry["ascii_name"], pool_df_ctry["ascii_addr"])]
    pool_ids = pool_df_ctry["entity_id"].astype(str).values

    print(f"  Pool loaded & normalized: {len(pool_df_ctry):,} records in {time.time()-t_load:.1f}s, RSS={rss_gb():.2f} GB")

    # Fit country-specific TF-IDF Vectorizer
    t_tfidf = time.time()
    tfidf = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 4),
        max_features=150000,
        min_df=cfg["min_df"],
        max_df=cfg["max_df"],
        sublinear_tf=True,
        dtype=np.float32
    )
    P_matrix = tfidf.fit_transform(pool_df_ctry["search_text"])
    print(f"  TF-IDF built: {P_matrix.shape} (nnz={P_matrix.nnz:,}) in {time.time()-t_tfidf:.1f}s, RSS={rss_gb():.2f} GB")

    # Query matching via 8 P-cores
    t_search = time.time()
    q_texts = ctry_q_df["search_text"].values
    q_ids = ctry_q_df["entity_id"].astype(str).values
    shard_candidates = {}
    shard_scores = {}

    _nb = (n_queries + BATCH_SIZE - 1) // BATCH_SIZE
    for b_start in pbar(range(0, n_queries, BATCH_SIZE), total=_nb, desc=f"match {ctry}"):
        b_end = min(b_start + BATCH_SIZE, n_queries)
        b_texts = q_texts[b_start:b_end]
        Q_batch = tfidf.transform(b_texts)

        top_mat = sp_matmul_topn(
            Q_batch, P_matrix.T,
            top_n=top_k,
            threshold=0.02,
            n_threads=NUM_P_CORES
        )

        for row_i in range(top_mat.shape[0]):
            q_idx = b_start + row_i
            qid = q_ids[q_idx]
            row = top_mat.getrow(row_i)
            indices, scs = row.indices, row.data
            
            if len(scs) > 1:
                order = np.argsort(-scs)
                indices, scs = indices[order], scs[order]
                
            shard_candidates[qid] = [pool_ids[ci] for ci in indices]
            shard_scores[qid] = list(scs)

    elapsed = time.time() - t_search
    mean_k = np.mean([len(c) for c in shard_candidates.values()])
    print(f"  Matched in {elapsed:.1f}s (mean K={mean_k:.1f})")

    # ---- v4 gated second pass (US recall lift; confident queries untouched) ----
    if ctry == "US":
        from rapidfuzz import fuzz as _fuzz
        _t2 = time.time()
        _low = [i for i, _qid in enumerate(q_ids)
                if ((shard_scores[_qid][0] if shard_scores[_qid] else 0.0) < 0.15
                    or len(shard_candidates[_qid]) < top_k // 2)]
        print(f"  Second pass: {len(_low)} low-confidence queries of {n_queries}")
        if _low:
            # (a) adaptive K: re-query low-confidence subset with top_n=60, lower threshold
            _Qlow = tfidf.transform(q_texts[_low])
            _top_low = sp_matmul_topn(_Qlow, P_matrix.T, top_n=60, threshold=0.02,
                                      n_threads=NUM_P_CORES)
            for _r, _qi in enumerate(_low):
                _qid = q_ids[_qi]
                _row = _top_low.getrow(_r)
                _idx, _sc = _row.indices, _row.data
                if len(_sc) > 1:
                    _o = np.argsort(-_sc)
                    _idx, _sc = _idx[_o], _sc[_o]
                _merged = dict(zip(shard_candidates[_qid], shard_scores[_qid]))
                for _ci, _ss in zip(_idx, _sc):
                    _pid = pool_ids[_ci]
                    if _pid not in _merged or _ss > _merged[_pid]:
                        _merged[_pid] = float(_ss)
                _items = sorted(_merged.items(), key=lambda kv: -kv[1])[:60]
                shard_candidates[_qid] = [k for k, _ in _items]
                shard_scores[_qid] = [v for _, v in _items]
            # (b) phonetic + house-number backfill, fuzz-ranked, capped at +10/query
            _num_re = re.compile(r"\b\d{1,6}\b")
            _pool_search = pool_df_ctry["search_text"].tolist()
            _pool_names = pool_df_ctry["ascii_name"].tolist()
            _pool_addrs = pool_df_ctry["ascii_addr"].tolist()
            _snd_index = collections.defaultdict(list)
            _num_index = collections.defaultdict(list)
            for _pi in range(len(pool_ids)):
                _ft = _pool_names[_pi].split(" ")[0] if _pool_names[_pi] else ""
                _snd_index[soundex(_ft)].append(_pi)
                for _nn in set(_num_re.findall(_pool_addrs[_pi])):
                    _num_index[_nn].append(_pi)
            for _kk in [k for k, v in _snd_index.items() if len(v) > 500]:
                del _snd_index[_kk]
            for _kk in [k for k, v in _num_index.items() if len(v) > 500]:
                del _num_index[_kk]
            _q_names = ctry_q_df["ascii_name"].tolist()
            _q_addrs = ctry_q_df["ascii_addr"].tolist()
            _q_texts = ctry_q_df["search_text"].tolist()
            for _qi in _low:
                _qid = q_ids[_qi]
                _have = set(shard_candidates[_qid])
                _qn = _q_names[_qi]
                _qs = soundex(_qn.split(" ")[0] if _qn else "")
                _pos = set(_snd_index.get(_qs, [])[:200])
                for _nn in set(_num_re.findall(_q_addrs[_qi])):
                    _pos.update(_num_index.get(_nn, [])[:200])
                _pos = [_p for _p in _pos if pool_ids[_p] not in _have]
                if not _pos:
                    continue
                _qt = _q_texts[_qi]
                _ranked = sorted(((_p, _fuzz.WRatio(_qt, _pool_search[_p])) for _p in _pos),
                                key=lambda t: -t[1])
                _add = [(_p, _s) for _p, _s in _ranked if _s >= 60][:10]
                if _add:
                    _merged = dict(zip(shard_candidates[_qid], shard_scores[_qid]))
                    for _p, _s in _add:
                        _merged.setdefault(pool_ids[_p], float(_s) / 100.0)
                    _items = sorted(_merged.items(), key=lambda kv: -kv[1])[:60]
                    shard_candidates[_qid] = [k for k, _ in _items]
                    shard_scores[_qid] = [v for _, v in _items]
            _mean2 = np.mean([len(shard_candidates[q]) for q in q_ids])
            print(f"  Second pass done in {time.time()-_t2:.1f}s (mean K={_mean2:.1f})")

    # Save shard cache
    with open(ckpt_path, "wb") as f:
        pickle.dump({"candidates": shard_candidates, "scores": shard_scores}, f)

    candidates_all.update(shard_candidates)
    scores_all.update(shard_scores)

    del pool_df_ctry, P_matrix, tfidf
    gc.collect()
    print(f"  Shard '{ctry}' finished in {time.time()-t_shard:.1f}s. RSS={rss_gb():.2f} GB")

# 5. Persist candidates to TSV
cand_tsv_path = OUT_DIR / "candidate_pairs.tsv"
write_id_list(cand_tsv_path, candidates_all, "candidate_entity_ids")
print(f"\nWrote candidate pairs: {cand_tsv_path} ({len(candidates_all):,} queries)")


=== Phase 2: Country-Split TF-IDF Blocking ===
Hardware: Pinned strictly to 12 P-cores. RSS: 2.77 GB
Target S1 queries: 20,000 (RUN_FULL=False)
Loaded 20,000 query records in 4.9s, RSS=2.81 GB
Countries to process: ['India', 'US']

--- Shard: 'India' (8,065 queries, top_k=90) ---
  LOCKED: loading candidates_v4_india_8065.pkl (frozen winner, no recompute)...

--- Shard: 'US' (11,935 queries, top_k=80) ---
  LOCKED: loading candidates_v5_us_11935.pkl (frozen winner, no recompute)...

Wrote candidate pairs: output-local\candidate_pairs.tsv (20,000 queries)


In [12]:
# === Phase 2 Blocking: Comprehensive Evaluation & Diagnostics ===
# Run this cell anytime to see per-country recall, candidate counts, and inspect sample hits/misses.

hits_by_country = {}
gt_by_country = {}
k_by_country = {}
sample_hits = []
sample_misses = []

# Map entity ID to country for fast lookup
s1_country_map = dict(zip(q_df["entity_id"].astype(str), q_df["country"]))

for qid in target_s1_list:
    ctry = s1_country_map.get(qid, "Unknown")
    cands = set(candidates_all.get(qid, []))
    true_matches = set(val_matches.get(qid, []))
    
    k_by_country.setdefault(ctry, []).append(len(cands))
    
    if true_matches:
        gt_by_country[ctry] = gt_by_country.get(ctry, 0) + len(true_matches)
        hit_set = cands & true_matches
        hits_by_country[ctry] = hits_by_country.get(ctry, 0) + len(hit_set)
        
        # Save up to 3 sample hits per country
        if len(hit_set) > 0 and sum(1 for h in sample_hits if h[2] == ctry) < 3:
            sample_hits.append((qid, list(hit_set)[0], ctry))
            
        # Save up to 3 sample misses per country
        missed_set = true_matches - cands
        if len(missed_set) > 0 and sum(1 for m in sample_misses if m[2] == ctry) < 3:
            sample_misses.append((qid, list(missed_set)[0], ctry))

print("\n" + "="*70)
print("           PHASE 2 BLOCKING: PER-COUNTRY RECALL REPORT")
print("="*70)
print(f"{'Country':<12} | {'Recall':<9} | {'Hits / Total GT':<18} | {'Mean K':<8} | {'Status'}")
print("-" * 70)

tot_hits, tot_gt, all_k = 0, 0, []
for ctry in sorted(gt_by_country.keys()):
    h = hits_by_country.get(ctry, 0)
    g = gt_by_country[ctry]
    rec = (h / g * 100) if g > 0 else 0.0
    mk = np.mean(k_by_country[ctry]) if ctry in k_by_country else 0.0
    tot_hits += h
    tot_gt += g
    all_k.extend(k_by_country.get(ctry, []))
    gate = 88.0 if ctry == "India" else 95.0
    status = " PASS (>=88%)" if (ctry == "India" and rec >= 88.0) else (" PASS (>=95%)" if (ctry != "India" and rec >= 95.0) else f" LOW (<{gate:.0f}%)")
    print(f"{ctry:<12} | {rec:6.2f}%   | {h:>7,} / {g:<7,}   | {mk:6.1f}   | {status}")

print("-" * 70)
tot_rec = (tot_hits / tot_gt * 100) if tot_gt > 0 else 0.0
overall_status = " PASS (>=92%)" if tot_rec >= 92.0 else " BELOW TARGET"
print(f"{'OVERALL':<12} | {tot_rec:6.2f}%   | {tot_hits:>7,} / {tot_gt:<7,}   | {np.mean(all_k):6.1f}   | {overall_status}")
print("=" * 70)

# Quick Inspection of Successes
print("\n--- SAMPLE RETRIEVED TRUE MATCHES ---")
for qid, mid, ctry in sample_hits:
    q_row = q_df[q_df["entity_id"].astype(str) == qid].iloc[0]
    print(f"[{ctry}] Q ({qid}): {q_row['business_name']} | {str(q_row['business_address'])[:50]}...")
    print(f"       -> RETRIEVED TRUE MATCH: {mid}")

# Quick Inspection of Misses (if any)
if sample_misses:
    print("\n--- SAMPLE MISSED TRUE MATCHES ---")
    for qid, mid, ctry in sample_misses:
        q_row = q_df[q_df["entity_id"].astype(str) == qid].iloc[0]
        print(f"[{ctry}] Q ({qid}): {q_row['business_name']} | {str(q_row['business_address'])[:50]}...")
        print(f"       -> MISSED TRUE MATCH: {mid}")
else:
    print("\nNo misses in evaluated sample!")



           PHASE 2 BLOCKING: PER-COUNTRY RECALL REPORT
Country      | Recall    | Hits / Total GT    | Mean K   | Status
----------------------------------------------------------------------
India        |  92.82%   |  25,624 / 27,605    |   90.0   |  PASS (>=88%)
US           |  95.03%   |  39,335 / 41,391    |   80.0   |  PASS (>=95%)
----------------------------------------------------------------------
OVERALL      |  94.15%   |  64,959 / 68,996    |   84.0   |  PASS (>=92%)

--- SAMPLE RETRIEVED TRUE MATCHES ---
[US] Q (S1-100007010): Quinlan Dental Associates | 1000 41st Avenue, MN, Columbia Heights, Unit Unit ...
       -> RETRIEVED TRUE MATCH: S3-324341187
[US] Q (S1-100013446): Mentor Family Partners Inc. | 8187 Johnnycake Ridge Road, Mentor, OH...
       -> RETRIEVED TRUE MATCH: S3-236069335
[US] Q (S1-100016578): Capital All Institutions P.C. | 6625 379th Court, North Branch, MN...
       -> RETRIEVED TRUE MATCH: S2-307909004
[India] Q (S1-10002196): Laxmi (India) Systech 

## Phase 2c — Multi-key union + IBF rank-merge challenger (experimental)
Builds 7 key indexes per country shard (exact name, significant tokens, name prefix, PIN, big numbers, house+locality, distinctive locality tokens), IBF-scores key candidates, and RRF-fuses with the LOCKED TF-IDF ranking, capped at top-50. Writes separate `candidate_pairs_v6.tsv` + v6 checkpoints — never touches the frozen v4/v5 live set. Gate to replace frozen: overall recall ≥95% AND mean K ≤50 (frozen: 94.15% @ K84). Expect ~10 min.

In [ ]:
# === Phase 2c: multi-key union + IBF rank-merge challenger (writes v6 only) ===
import math
import time
from collections import defaultdict as _dd3
t2c = time.time()

K_TOP = 50
RRF_K = 60
W = {"exact": 3.0, "pin": 2.5, "bignum": 2.0, "house": 1.5,
     "token": 1.0, "prefix": 1.0, "loc": 1.0}
CAP = {"exact": 500, "pin": 500, "bignum": 500, "house": 250,
       "prefix": 250, "token": 150, "loc": 150}
STOP2 = set(LEGAL_SUFFIX) | {"the", "and", "for", "with", "services", "group"}
_pin_re2 = re.compile(r"(?<!\d)(\d{5,6})(?!\d)")
_bignum_re2 = re.compile(r"\b\d{4,}\b")
_housenum_re2 = re.compile(r"\b\d{1,3}\b")

def keys_of(name, addr):
    ntoks = [t for t in name.split() if len(t) >= 3 and t not in STOP2]
    atoks = [t for t in addr.split() if len(t) >= 3]
    compact = name.replace(" ", "")
    out = {"exact": [name] if name else [],
           "token": ntoks,
           "prefix": [compact[:4]] if len(compact) >= 4 else [],
           "pin": _pin_re2.findall(addr),
           "bignum": list(set(_bignum_re2.findall(name + " " + addr))),
           "house": [],
           "loc": [t for t in atoks if len(t) >= 6]}
    _hn = _housenum_re2.findall(addr)
    if _hn and atoks:
        out["house"] = [_hn[0] + "|" + atoks[0]]
    return out

cand6_all, score6_all = {}, {}
for _ctry in sorted(q_df["country"].unique()):
    _t = time.time()
    _qd = q_df[q_df["country"] == _ctry].reset_index(drop=True)
    _nq = len(_qd)
    if _nq == 0:
        continue
    print(f"--- 2c shard '{_ctry}' ({_nq:,} queries) ---", flush=True)
    _recs = []
    for _sf in ["train_source2.tsv", "train_source3.tsv"]:
        for _ch in pd.read_csv(DATA_ROOT / "train" / _sf, sep="\t", chunksize=250000):
            _ch["country"] = _ch["country"].fillna("").str.strip()
            _h = _ch[_ch["country"] == _ctry]
            if len(_h):
                _recs.append(_h[["entity_id", "business_name", "business_address"]])
    _pdf = pd.concat(_recs, ignore_index=True).drop_duplicates("entity_id")
    del _recs
    gc.collect()
    _pn = _pdf["business_name"].fillna("").map(normalize_ascii).tolist()
    _pa = _pdf["business_address"].fillna("").map(normalize_ascii).tolist()
    _pids = _pdf["entity_id"].astype(str).values
    del _pdf
    gc.collect()
    print(f"  pool: {len(_pids):,} in {time.time()-_t:.0f}s rss={rss_gb():.2f} GB", flush=True)
    _index = {kt: _dd3(list) for kt in W}
    for _pi in pbar(range(len(_pids)), total=len(_pids), desc=f"2c index {ctry}"):
        _kk = keys_of(_pn[_pi], _pa[_pi])
        for _kt, _kl in _kk.items():
            for _k in _kl:
                _index[_kt][_k].append(_pi)
    del _pn, _pa
    gc.collect()
    for _kt in W:
        for _k in [k for k, v in _index[_kt].items() if len(v) > CAP[_kt]]:
            del _index[_kt][_k]
    print(f"  index keys: {sum(len(d) for d in _index.values()):,} rss={rss_gb():.2f} GB", flush=True)
    _qn = _qd["ascii_name"].tolist()
    _qa = _qd["ascii_addr"].tolist()
    _qids = _qd["entity_id"].astype(str).values
    for _qi in pbar(range(_nq), total=_nq, desc=f"2c rank {ctry}"):
        _qid = _qids[_qi]
        _qk = keys_of(_qn[_qi], _qa[_qi])
        _sc = {}
        for _kt, _kl in _qk.items():
            _d = _index[_kt]
            for _k in _kl:
                _bkt = _d.get(_k)
                if not _bkt:
                    continue
                _w = W[_kt]/math.log2(2 + len(_bkt))
                for _pi in _bkt:
                    _sc[_pi] = _sc.get(_pi, 0.0) + _w
        _ibf_rank = {}
        _ranked = sorted(_sc.items(), key=lambda kv: -kv[1])[:100]
        for _r, (_pi, _s) in enumerate(_ranked):
            _ibf_rank[_pi] = _r
        _lock = candidates_all.get(_qid, [])
        _fused = {}
        for _r, _pid in enumerate(_lock):
            _fused[_pid] = _fused.get(_pid, 0.0) + 1.0/(RRF_K + _r)
        for _pi, _r in _ibf_rank.items():
            _pid = _pids[_pi]
            _fused[_pid] = _fused.get(_pid, 0.0) + 1.0/(RRF_K + _r)
        _top = sorted(_fused.items(), key=lambda kv: -kv[1])[:K_TOP]
        cand6_all[_qid] = [k for k, _ in _top]
        score6_all[_qid] = [v for _, v in _top]
    del _index
    gc.collect()
    print(f"  shard done in {time.time()-_t:.0f}s rss={rss_gb():.2f} GB", flush=True)

write_id_list(OUT_DIR / "candidate_pairs_v6.tsv", cand6_all, "candidate_entity_ids")
with open(OUT_DIR / "checkpoints" / "candidates_v6_combined_20000.pkl", "wb") as _f:
    pickle.dump({"candidates": cand6_all, "scores": score6_all}, _f)
_h6, _g6, _k6 = 0, 0, []
for _qid in target_s1_list:
    _t = set(val_matches.get(_qid, []))
    _c = set(cand6_all.get(_qid, []))
    _k6.append(len(_c))
    if _t:
        _g6 += len(_t)
        _h6 += len(_c & _t)
import numpy as _np6
print(f"v6 challenger: recall={_h6/_g6*100:.2f}% ({_h6:,}/{_g6:,}) meanK={_np6.mean(_k6):.1f}", flush=True)
print("GATE vs frozen (94.15% @ K84):", "REPLACE" if (_h6/_g6 >= 0.95 and _np6.mean(_k6) <= 50) else "KEEP FROZEN", flush=True)
print(f"Phase 2c done in {time.time()-t2c:.0f}s (live v4/v5 set untouched)", flush=True)


## Phase 3 — LightGBM matcher on frozen candidates (local CPU)
Trains on GT positives + 1:6 hard negatives from the LOCKED v4/v5 `candidates_all`, GroupKFold by S1 entity, LightGBM baseline, dual thresholds on entity-macro-F0.5. Writes `lgbm_model.txt` + `model_config.json` to `output-local/`. Expect ~30-45 min (feature build dominates).

In [ ]:
# === Phase 3a: pairwise feature matrix over frozen candidates (local CPU) ===
import time
t3a = time.time()
rng = np.random.RandomState(42)
from rapidfuzz import fuzz as _fuzz

cand_universe = set()
for _v in candidates_all.values():
    cand_universe.update(_v)
print(f"candidate universe: {len(cand_universe):,} ids", flush=True)
pool_parts = []
for _sf in ["train_source2.tsv", "train_source3.tsv"]:
    for _ch in pd.read_csv(DATA_ROOT / "train" / _sf, sep="\t", chunksize=250000):
        _hit = _ch[_ch["entity_id"].astype(str).isin(cand_universe)]
        if len(_hit):
            pool_parts.append(_hit[["entity_id", "business_name", "business_address", "country"]])
pool = pd.concat(pool_parts, ignore_index=True).drop_duplicates("entity_id")
del pool_parts
gc.collect()
pool["ascii_name"] = pool["business_name"].fillna("").map(normalize_ascii)
pool["ascii_addr"] = pool["business_address"].fillna("").map(normalize_ascii)
print(f"pool rows kept: {len(pool):,} rss={rss_gb():.2f} GB", flush=True)
pidx = {}
_ppin = {}
for _r in pool.itertuples():
    pidx[_r.entity_id] = _r
    _ppin[str(_r.entity_id)] = extract_pin(_r.ascii_addr)
_qpin = {str(r.entity_id): extract_pin(r.ascii_addr) for r in q_df.itertuples()}
s1x = {r.entity_id: r for r in q_df.itertuples()}

_house_re = re.compile(r"\b\d{1,6}\b")
def _char3(s):
    s = s.replace(" ", "")
    return {s[i:i+3] for i in range(max(0, len(s)-2))} or {""}

def _is_missing_addr(x):
    x = str(x).strip().lower()
    return (not x) or x in ("nan", "none", "null", "-")

def pair_feats(a_name, a_addr, b_name, b_addr, tfidf_s, same_ctry, is_s2, addr_missing, pa_pin, pb_pin):
    la, lb = len(a_name), len(b_name)
    ca, cb = _char3(a_name), _char3(b_name)
    aw, bw = a_addr.split(), b_addr.split()
    saw, sbw = set(aw), set(bw)
    return [tfidf_s,
            _fuzz.WRatio(a_name, b_name)/100.0,
            _fuzz.token_set_ratio(a_name, b_name)/100.0,
            _fuzz.token_sort_ratio(a_name, b_name)/100.0,
            _fuzz.token_sort_ratio(a_addr, b_addr)/100.0,
            (min(la, lb)/max(la, lb)) if max(la, lb) else 0.0,
            len(ca & cb)/max(1, len(ca | cb)),
            1.0 if (set(_house_re.findall(a_addr)) & set(_house_re.findall(b_addr))) else 0.0,
            1.0 if same_ctry else 0.0,
            1.0 if is_s2 else 0.0,
            1.0 if addr_missing else 0.0,
            _fuzz.token_set_ratio(a_addr, b_addr)/100.0,
            len(saw & sbw)/max(1, len(saw | sbw)),
            _fuzz.partial_ratio(a_name, b_name)/100.0,
            _fuzz.partial_ratio(a_addr, b_addr)/100.0,
            1.0 if a_name == b_name else 0.0,
            1.0 if (pa_pin and pa_pin == pb_pin) else 0.0,
            1.0 if (pa_pin and pb_pin and pa_pin != pb_pin) else 0.0]

FEATS = ["tfidf", "name_wratio", "name_set", "name_sort", "addr_sort", "len_ratio", "char3_jac",
         "num_match", "country_match", "source_is_s2", "address_missing",
         "addr_set", "addr_word_jac", "name_partial", "addr_partial", "name_exact",
         "pin_match", "pin_mismatch"]

Xa, ya, qa, ca, ma = [], [], [], [], []
for _qid in pbar(target_s1_list, total=len(target_s1_list), desc="features"):
    _a = s1x.get(_qid)
    if _a is None:
        continue
    _truth = set(val_matches.get(_qid, []))
    _cands = candidates_all.get(_qid, [])
    _sc = scores_all.get(_qid, [0.0]*len(_cands))
    _smap = dict(zip(_cands, _sc))
    _pos = [c for c in _cands if c in _truth]
    _negpool = [c for c in _cands if c not in _truth]
    if _truth:
        _k = min(len(_negpool), 6*max(1, len(_pos)))
        _negset = set(rng.choice(_negpool, _k, replace=False).tolist()) if _k else set()
    else:
        _k = min(len(_negpool), 12)
        _negset = set(rng.choice(_negpool, _k, replace=False).tolist()) if _k else set()
    for _c in _cands:
        _b = pidx.get(_c)
        if _b is None:
            continue
        _amiss = _is_missing_addr(_a.business_address) or _is_missing_addr(_b.business_address)
        Xa.append(pair_feats(_a.ascii_name, _a.ascii_addr, _b.ascii_name, _b.ascii_addr,
                             float(_smap.get(_c, 0.0)), str(_a.country) == str(_b.country),
                             str(_c).startswith("S2-"), _amiss,
                             _qpin.get(_qid, ""), _ppin.get(str(_c), "")))
        ya.append(1 if _c in _truth else 0)
        qa.append(_qid)
        ca.append(_c)
        ma.append(1 if (_c in _truth or _c in _negset) else 0)
X_all = np.array(Xa, dtype=np.float32)
y_all = np.array(ya, dtype=np.int8)
q_all = np.array(qa, dtype=object)
c_all = np.array(ca, dtype=object)
tr_mask = np.array(ma, dtype=bool)
del Xa, ya, qa, ca, ma
gc.collect()
print(f"pairs: {len(y_all):,} (pos={int(y_all.sum()):,}, train_mask={int(tr_mask.sum()):,}) in {time.time()-t3a:.0f}s, rss={rss_gb():.2f} GB", flush=True)


In [ ]:
# === Phase 3b: GroupKFold LightGBM by S1 entity + OOF over all candidates ===
import os
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
t3b = time.time()
NT = int(os.cpu_count() or 8)
LP = dict(objective="binary", learning_rate=0.05, num_leaves=63, max_depth=7,
          min_child_samples=80, subsample=0.8, colsample_bytree=0.8,
          reg_lambda=1.0, n_estimators=2000, n_jobs=NT, verbose=-1)
gkf = GroupKFold(n_splits=3)
oof_all = np.zeros(len(y_all))
best_iters = []
for _fi, (_tr, _va) in enumerate(gkf.split(X_all, y_all, q_all)):
    _m = lgb.LGBMClassifier(**LP)
    _m.fit(X_all[_tr][tr_mask[_tr]], y_all[_tr][tr_mask[_tr]],
           eval_X=X_all[_va], eval_y=y_all[_va],
           callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_all[_va] = _m.predict_proba(X_all[_va])[:, 1]
    best_iters.append(int(_m.best_iteration_))
    print(f"fold {_fi}: best_iter={_m.best_iteration_} val_pairs={len(_va):,}", flush=True)
print(f"OOF done in {time.time()-t3b:.0f}s, best_iters={best_iters}", flush=True)


In [ ]:
# === Phase 3c: dual thresholds on OOF + refit + artifacts ===
import json
import os as _os
from collections import defaultdict as _dd
t3c = time.time()
_sid_by_q = _dd(list)
_sco_by_q = _dd(list)
for _qq, _cc, _ss in zip(q_all, c_all, oof_all):
    _sid_by_q[_qq].append(_cc)
    _sco_by_q[_qq].append(float(_ss))
_truth = {q: val_matches.get(q, []) for q in target_s1_list}
best_f, best_ts, best_tm, best_key = -1.0, 0.5, 0.5, None
for _ts in [round(float(x), 2) for x in np.arange(0.30, 0.91, 0.05)]:
    for _tm in [round(float(x), 2) for x in np.arange(0.30, 0.91, 0.05)]:
        _pred = {}
        for _q in target_s1_list:
            _ss = _sco_by_q.get(_q, [])
            if not _ss or max(_ss) < _ts:
                _pred[_q] = []
            else:
                _pred[_q] = [_c for _c, _s in zip(_sid_by_q[_q], _ss) if _s >= _tm]
        _f = macro_f05(_truth, _pred)
        _key = (round(_f, 9), round(_ts + _tm, 9))
        if best_key is None or _key > best_key:
            best_f, best_ts, best_tm, best_key = _f, _ts, _tm, _key
print(f"tuned: T_singleton={best_ts} T_match={best_tm} OOF macro-F0.5={best_f:.4f} (skeleton was 0.088)", flush=True)
_nest = int(np.median(best_iters))
_final = lgb.LGBMClassifier(**{**LP, "n_estimators": _nest})
_final.fit(X_all[tr_mask], y_all[tr_mask])
_final.booster_.save_model(str(OUT_DIR / "lgbm_model.txt"))
_cfg = {"features": FEATS,
        "lightgbm": {k: (v if isinstance(v, (int, float, str)) else str(v)) for k, v in LP.items()},
        "n_estimators_refit": _nest, "T_singleton": best_ts, "T_match": best_tm,
        "oof_macro_f05": round(float(best_f), 4), "best_iters": best_iters,
        "train_pairs": int(tr_mask.sum()), "all_pairs": int(len(y_all)), "seed": 42,
        "candidates": {"India": "candidates_v4_india_8065.pkl", "US": "candidates_v5_us_11935.pkl"}}
(OUT_DIR / "model_config.json").write_text(json.dumps(_cfg, indent=1))
print("saved lgbm_model.txt + model_config.json", flush=True)
_imp = sorted(zip(FEATS, _final.feature_importances_), key=lambda t: -t[1])
print("gain importance:", [(f, int(v)) for f, v in _imp], flush=True)
print(f"Phase 3c done in {time.time()-t3c:.0f}s", flush=True)


## Phase 5-smoke — Validate the test-blocking path (~10 min, run first)
Runs the frozen configs over the first 2,000 test queries with real per-shard pools. Must print the concat gate PASS before any full launch. Writes only to `test_parts_smoke/` + `candidate_pairs_test_smoke.tsv`.

In [31]:
# === Phase 5-smoke: validate test-blocking path on 2k queries (run first) ===
SMOKE_N = 2000
SMOKE_TOPK = {"India": 90, "US": 80, "__default__": 40}  # same frozen values as the full run
SMOKE_PARTS = OUT_DIR / "test_parts_smoke"
SMOKE_PARTS.mkdir(parents=True, exist_ok=True)
_tsm = time.time()
_sm = []
for _ch in pd.read_csv(DATA_ROOT / "test" / "test_source1.tsv", sep="\t", chunksize=250000,
                       usecols=["entity_id", "business_name", "business_address", "country"]):
    _ch["country"] = _ch["country"].fillna("").str.strip()
    _sm.append(_ch)
    if sum(len(x) for x in _sm) >= SMOKE_N:
        break
_sq = pd.concat(_sm, ignore_index=True).head(SMOKE_N)
del _sm
gc.collect()
_sq["ascii_name"] = _sq["business_name"].fillna("").map(normalize_ascii)
_sq["ascii_addr"] = _sq["business_address"].fillna("").map(normalize_ascii)
print(f"smoke queries: {len(_sq):,} countries={sorted(_sq['country'].unique())}", flush=True)
for _ctry in sorted(_sq["country"].unique()):
    _cfg = COUNTRY_CONFIGS.get(_ctry, COUNTRY_CONFIGS["__default__"])
    _topk = SMOKE_TOPK.get(_ctry, SMOKE_TOPK["__default__"])
    _tfn = _cfg["text_fn"]
    _cq = _sq[_sq["country"] == _ctry].reset_index(drop=True)
    _pr = []
    for _sf in ["test_source2.tsv", "test_source3.tsv"]:
        for _ch in pbar(pd.read_csv(DATA_ROOT / "test" / _sf, sep="\t", chunksize=250000), desc=f"pool {_ctry} {_sf}"):
            _ch["country"] = _ch["country"].fillna("").str.strip()
            _h = _ch[_ch["country"] == _ctry]
            if len(_h):
                _pr.append(_h[["entity_id", "business_name", "business_address"]])
    _pp = pd.concat(_pr, ignore_index=True).drop_duplicates("entity_id")
    del _pr
    gc.collect()
    print("  normalizing pool text...", flush=True)
    _pp["search_text"] = [_tfn(normalize_ascii(n), normalize_ascii(a))
                          for n, a in zip(_pp["business_name"].fillna(""), _pp["business_address"].fillna(""))]
    _pids = _pp["entity_id"].astype(str).values
    _vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), max_features=150000,
                           min_df=_cfg["min_df"], max_df=_cfg["max_df"],
                           sublinear_tf=True, dtype=np.float32)
    print("  fitting TF-IDF...", flush=True)
    _Pm = _vec.fit_transform(_pp["search_text"])
    del _pp
    gc.collect()
    _qids = _cq["entity_id"].astype(str).values
    _qtx = [_tfn(n, a) for n, a in zip(_cq["ascii_name"], _cq["ascii_addr"])]
    _rows = []
    _nsm = (len(_cq) + 5000 - 1) // 5000
    for _s in pbar(range(0, len(_cq), 5000), total=_nsm, desc=f"smoke {ctry}"):
        _Qb = _vec.transform(_qtx[_s:_s+5000])
        _Tm = sp_matmul_topn(_Qb, _Pm.T, top_n=_topk, threshold=0.02, n_threads=NUM_P_CORES)
        for _ri in range(_Tm.shape[0]):
            _row = _Tm.getrow(_ri)
            _ii, _ss = _row.indices, _row.data
            if len(_ss) > 1:
                _oo = np.argsort(-_ss)
                _ii = _ii[_oo]
            _rows.append((_qids[_s+_ri], ",".join(_pids[_c] for _c in _ii)))
    pd.DataFrame(_rows, columns=["source1_entity_id", "candidate_entity_ids"]).to_csv(
        SMOKE_PARTS / f"test_{_ctry}_smoke.tsv", sep="\t", index=False, encoding="utf-8")
    del _Pm, _vec
    gc.collect()
    print(f"  smoke shard '{_ctry}': {len(_rows):,} rows", flush=True)
_final = OUT_DIR / "candidate_pairs_test_smoke.tsv"
_nrows, _seen, _dup = 0, set(), 0
with open(_final, "w", encoding="utf-8") as _fo:
    _fo.write("source1_entity_id\tcandidate_entity_ids\n")
    for _part in sorted(SMOKE_PARTS.glob("test_*_smoke.tsv")):
        with open(_part, encoding="utf-8") as _fi:
            next(_fi)
            for _ln in _fi:
                _sid = _ln.split("\t", 1)[0]
                if _sid in _seen:
                    _dup += 1
                _seen.add(_sid)
                _fo.write(_ln)
                _nrows += 1
print(f"smoke concat: rows={_nrows:,} (queries={len(_sq):,}) dup_S1={_dup}", flush=True)
assert _nrows == len(_sq) and _dup == 0, "smoke coverage check failed"
print(f"SMOKE GATE: PASS in {(time.time()-_tsm)/60:.1f} min \u2014 full launch cleared", flush=True)


smoke queries: 2,000 countries=['France', 'India', 'US']
  smoke shard 'France': 283 rows
  smoke shard 'India': 933 rows
  smoke shard 'US': 784 rows
smoke concat: rows=2,000 (queries=2,000) dup_S1=0
SMOKE GATE: PASS in 13.9 min — full launch cleared


## Phase 5 — Full TEST blocking (resumable, chunked, memory-safe)
Runs the frozen configs over all 1.73M test S1 queries (~26h). Each 25k-query chunk writes its own part-TSV immediately, so any interruption resumes in seconds with zero redo. Final concat verifies one row per test S1. Run the p5-smoke cell first (must print SMOKE GATE: PASS), then launch with `RUN_TEST=True`.

In [ ]:
# === Phase 5: full TEST blocking (resumable per chunk; safe to interrupt) ===
RUN_TEST = True  # flip True to launch
TEST_SHARDS = ["India"]  # multi-device: e.g. ["India"] here, ["US", "France"] there; None = all shards
TEST_TOPK = {"India": 90, "US": 80, "__default__": 40}  # frozen tuned values; trim only at known recall cost
TEST_CHUNK = 25000
TEST_CHUNK_START = 0  # multi-device split: e.g. dev A 0-35, dev B 35-70 (chunk idx, per shard)
TEST_CHUNK_END = None  # None = last chunk; partial ranges skip the final concat (merge machine does it)
TEST_BATCH = 5000
TEST_THRESH = 0.02
TEST_TAG = "full"
TEST_PARTS = OUT_DIR / "test_parts_full"
TEST_PARTS.mkdir(parents=True, exist_ok=True)
RATE = {"India": 0.075, "US": 0.041, "__default__": 0.015}  # measured s/query, scaled by pool
def _git_head():
    import subprocess
    for _d in [Path.cwd(), Path.cwd() / ".."]: 
        try:
            _r = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                                text=True, cwd=str(_d), timeout=15)
            if _r.returncode == 0:
                return _r.stdout.strip()
        except Exception:
            pass
    return "unknown"

if not RUN_TEST:
    print("RUN_TEST=False \u2014 set True to launch (p5-smoke cell must have printed SMOKE GATE: PASS first).")
else:
    t5 = time.time()
    _tq = []
    for _ch in pd.read_csv(DATA_ROOT / "test" / "test_source1.tsv", sep="\t",
                           chunksize=250000, usecols=["entity_id", "business_name", "business_address", "country"]):
        _ch["country"] = _ch["country"].fillna("").str.strip()
        _tq.append(_ch)
    _tqdf = pd.concat(_tq, ignore_index=True)
    del _tq
    gc.collect()
    _tqdf = _tqdf  # full test set (smoke lives in the p5-smoke cell)
    pass
    _tqdf["ascii_name"] = _tqdf["business_name"].fillna("").map(normalize_ascii)
    _tqdf["ascii_addr"] = _tqdf["business_address"].fillna("").map(normalize_ascii)
    _est = sum(len(_tqdf[_tqdf["country"] == c]) * RATE.get(c, RATE["__default__"])
               for c in _tqdf["country"].unique())
    print(f"test queries: {len(_tqdf):,} | est. match time ~{_est/3600:.1f}h | tag={TEST_TAG}", flush=True)
    _shards = [c for c in sorted(_tqdf["country"].unique()) if TEST_SHARDS is None or c in TEST_SHARDS]
    print(f"shards on this device: {_shards}", flush=True)
    for _ctry in _shards:
        _ts = time.time()
        _cfg = COUNTRY_CONFIGS.get(_ctry, COUNTRY_CONFIGS["__default__"])
        _topk = TEST_TOPK.get(_ctry, TEST_TOPK["__default__"])
        _tfn = _cfg["text_fn"]
        _cq = _tqdf[_tqdf["country"] == _ctry].reset_index(drop=True)
        _nq = len(_cq)
        print(f"--- test shard '{_ctry}' ({_nq:,} queries, top_k={_topk}) ---", flush=True)
        _pr = []
        for _sf in ["test_source2.tsv", "test_source3.tsv"]:
            for _ch in pbar(pd.read_csv(DATA_ROOT / "test" / _sf, sep="\t", chunksize=250000), desc=f"pool {_ctry} {_sf}"):
                _ch["country"] = _ch["country"].fillna("").str.strip()
                _h = _ch[_ch["country"] == _ctry]
                if len(_h):
                    _pr.append(_h[["entity_id", "business_name", "business_address"]])
        _pp = pd.concat(_pr, ignore_index=True).drop_duplicates("entity_id")
        del _pr
        gc.collect()
        print("  normalizing pool text...", flush=True)
        _pp["search_text"] = [_tfn(normalize_ascii(n), normalize_ascii(a))
                              for n, a in zip(_pp["business_name"].fillna(""), _pp["business_address"].fillna(""))]
        _pids = _pp["entity_id"].astype(str).values
        _vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), max_features=150000,
                               min_df=_cfg["min_df"], max_df=_cfg["max_df"],
                               sublinear_tf=True, dtype=np.float32)
        print("  fitting TF-IDF...", flush=True)
        _Pm = _vec.fit_transform(_pp["search_text"])
        del _pp
        gc.collect()
        print(f"  pool TF-IDF: {_Pm.shape} rss={rss_gb():.2f} GB", flush=True)
        _qids = _cq["entity_id"].astype(str).values
        _qtx = [_tfn(n, a) for n, a in zip(_cq["ascii_name"], _cq["ascii_addr"])]
        _nch = (_nq + TEST_CHUNK - 1) // TEST_CHUNK
        _cs, _ce = (TEST_CHUNK_START or 0), (TEST_CHUNK_END or _nch)
        _ct0 = time.time()
        for _ci in range(_cs, min(_ce, _nch)):
            _a, _b = _ci*TEST_CHUNK, min((_ci+1)*TEST_CHUNK, _nq)
            _part = TEST_PARTS / f"test_{_ctry}_c{_ci:03d}.tsv"
            if _part.exists():
                try:
                    _nlines = sum(1 for _ in open(_part, encoding="utf-8")) - 1
                except OSError:
                    _nlines = -1
                if _nlines == _b - _a:
                    print(f"  chunk {_ci+1}/{_nch}: resume-skip ({_b-_a:,} rows present)", flush=True)
                    continue
                print(f"  chunk {_ci+1}/{_nch}: incomplete part ({_nlines} vs {_b-_a} rows) \u2014 recomputing", flush=True)
            _rows, _tcc = [], time.time()
            _nbt = (_b - _a + TEST_BATCH - 1) // TEST_BATCH
            for _s in pbar(range(_a, _b, TEST_BATCH), total=_nbt, desc=f"chunk {_ci+1}"):
                _Qb = _vec.transform(_qtx[_s:min(_s+TEST_BATCH, _b)])
                _Tm = sp_matmul_topn(_Qb, _Pm.T, top_n=_topk, threshold=TEST_THRESH, n_threads=NUM_P_CORES)
                for _ri in range(_Tm.shape[0]):
                    _row = _Tm.getrow(_ri)
                    _ii, _ss = _row.indices, _row.data
                    if len(_ss) > 1:
                        _oo = np.argsort(-_ss)
                        _ii = _ii[_oo]
                    _rows.append((_qids[_s+_ri], ",".join(_pids[_c] for _c in _ii)))
            pd.DataFrame(_rows, columns=["source1_entity_id", "candidate_entity_ids"]).to_csv(
                _part, sep="\t", index=False, encoding="utf-8")
            _dt = time.time() - _tcc
            _elapsed_chunks = time.time() - _ct0
            _left = (_ce - _ci - 1) * _elapsed_chunks / max(1, _ci - _cs + 1)
            print(f"  chunk {_ci+1}/{_nch}: wrote {len(_rows):,} rows in {_dt:.0f}s (ETA shard {_left/3600:.1f}h)", flush=True)
        del _Pm, _vec
        gc.collect()
        print(f"  shard '{_ctry}' done in {(time.time()-_ts)/3600:.1f}h", flush=True)
    _is_full = (TEST_SHARDS is None) and (not TEST_CHUNK_START) and (TEST_CHUNK_END is None)
    if _is_full:
        _final = OUT_DIR / f"candidate_pairs_test_{TEST_TAG}.tsv"
        _nrows, _seen, _dup = 0, set(), 0
        with open(_final, "w", encoding="utf-8") as _fo:
            _fo.write("source1_entity_id\tcandidate_entity_ids\n")
            for _part in sorted(TEST_PARTS.glob("test_*.tsv")):
                with open(_part, encoding="utf-8") as _fi:
                    next(_fi)
                    for _ln in _fi:
                        _sid = _ln.split("\t", 1)[0]
                        if _sid in _seen:
                            _dup += 1
                        _seen.add(_sid)
                        _fo.write(_ln)
                        _nrows += 1
        print(f"concat: {_final.name} rows={_nrows:,} (queries={len(_tqdf):,}) dup_S1={_dup}", flush=True)
        assert _nrows == len(_tqdf) and _dup == 0, "concat coverage check failed"
        _manifest = {"tag": TEST_TAG, "topk": TEST_TOPK, "threshold": TEST_THRESH, "chunk": TEST_CHUNK,
                     "commit": _git_head(), "rows": _nrows, "seconds": round(time.time()-t5), "rss_gb": round(rss_gb(), 2)}
        (OUT_DIR / f"test_manifest_{TEST_TAG}.json").write_text(__import__("json").dumps(_manifest, indent=1))
        print(f"Phase 5 done in {(time.time()-t5)/3600:.1f}h \u2014 manifest written", flush=True)
    else:
        print("partial run \u2014 copy test_parts_full/*.tsv to the merge machine and run the p5-merge cell")



test queries: 1,732,544 | est. match time ~25.5h | tag=full
shards on this device: ['India']
--- test shard 'India' (809,986 queries, top_k=90) ---


pool India test_source2.tsv: 20it [00:08,  2.24it/s]
pool India test_source3.tsv: 21it [00:08,  2.38it/s]


  normalizing pool text...
  fitting TF-IDF...
  pool TF-IDF: (4717565, 150000) rss=9.78 GB


chunk 1: 100%|██████████| 5/5 [28:31<00:00, 342.39s/it]


  chunk 1/33: wrote 25,000 rows in 1712s (ETA shard 15.2h)


chunk 2: 100%|██████████| 5/5 [27:16<00:00, 327.22s/it]


  chunk 2/33: wrote 25,000 rows in 1636s (ETA shard 14.4h)


chunk 3: 100%|██████████| 5/5 [27:45<00:00, 333.19s/it]


  chunk 3/33: wrote 25,000 rows in 1666s (ETA shard 13.9h)


chunk 4:  60%|██████    | 3/5 [17:23<11:40, 350.34s/it]

## Phase 5-merge — Combine multi-device parts (merge machine only)
Copy every device's `test_parts_full/*.tsv` into this machine's `test_parts_full/`, then run. Verifies one row per test S1 with zero duplicate S1 rows, writes `candidate_pairs_test_full.tsv` + manifest (both manifests' configs must match before trusting the file).

In [ ]:
# === Phase 5-merge: concat + verify multi-device parts (merge machine) ===
import json as _json
MERGE_FROM = OUT_DIR / "test_parts_full"  # copy all devices' test_*.tsv here first
_tq_n = 0
for _ch in pd.read_csv(DATA_ROOT / "test" / "test_source1.tsv", sep="\t", chunksize=500000,
                       usecols=["entity_id"]):
    _tq_n += len(_ch)
_final = OUT_DIR / "candidate_pairs_test_full.tsv"
_nrows, _seen, _dup = 0, set(), 0
with open(_final, "w", encoding="utf-8") as _fo:
    _fo.write("source1_entity_id\tcandidate_entity_ids\n")
    for _part in sorted(MERGE_FROM.glob("test_*.tsv")):
        with open(_part, encoding="utf-8") as _fi:
            next(_fi)
            for _ln in _fi:
                _sid = _ln.split("\t", 1)[0]
                if _sid in _seen:
                    _dup += 1
                _seen.add(_sid)
                _fo.write(_ln)
                _nrows += 1
print(f"merge concat: rows={_nrows:,} (test queries={_tq_n:,}) dup_S1={_dup}", flush=True)
assert _nrows == _tq_n and _dup == 0, "merge coverage check failed"
(OUT_DIR / "test_manifest_full.json").write_text(_json.dumps(
    {"merged_rows": _nrows, "test_queries": _tq_n, "commit": None}, indent=1))
print("MERGE GATE: PASS \u2014 candidate_pairs_test_full.tsv ready for Phase-5 scoring", flush=True)


## Phase 3 — OpenVINO transformer inference (Arc 140T iGPU)
OpenVINO accelerates *inference only*: fine-tune the bi-encoder once on a cloud GPU, export to OpenVINO IR, score candidates here on `GPU` (Arc 140T) with CPU fallback. FAISS/pandas/sklearn/LightGBM stay on CPU regardless.

In [ ]:
try:
    import openvino as ov
    core = ov.Core()
    print("openvino:", ov.__version__, "devices:", core.available_devices)
except Exception as e:
    core = None
    print("openvino unavailable:", type(e).__name__, e)

In [ ]:
# Export once (needs the fine-tuned encoder + GPU-box training first):
#   from optimum.intel import OVModelForFeatureExtraction
#   m = OVModelForFeatureExtraction.from_pretrained("sentence-transformers/all-MiniLM-L6-v2", export=True)
#   m.save_pretrained(MODEL_DIR)  # openvino_model.xml + tokenizer

if core is None or not MODEL_DIR.exists():
    print("OV stub: no exported model yet — fine-tune on a GPU box, export to models/minilm-ov, re-run.")
else:
    from optimum.intel import OVModelForFeatureExtraction
    from transformers import AutoTokenizer
    device = "GPU" if "GPU" in core.available_devices else "CPU"
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    om = OVModelForFeatureExtraction.from_pretrained(MODEL_DIR, device=device)
    a = tok("Lumay Boral [SEP] 1056 Belden Avenue Akron OH", return_tensors="pt")
    b = tok("Lumay Boral Inc [SEP] 1056 Belden Ave Akon Ohio", return_tensors="pt")
    ea, eb = om(**a).last_hidden_state.mean(1).detach().numpy(), om(**b).last_hidden_state.mean(1).detach().numpy()
    cos = float(ea @ eb.T / (np.linalg.norm(ea) * np.linalg.norm(eb)))
    print(f"device={device} pair-cosine={cos:.3f} rss={rss_gb():.2f} GB")

## Phase 3 / 4 / 5 — Roadmap (+ competitive upgrades, plan §6)
- **Phase 3:** extend `pair_score` into pairwise features + LightGBM on val candidates (CPU, all cores); bi-encoder fine-tune on cloud GPU, inference here via the OpenVINO cell above.
- **Phase 4:** DUAL thresholds (`T_singleton`, `T_match`) on val macro-F0.5 (expect ~0.6–0.8); `max_score < tau` → empty list.
- **Phase 5:** chunked test inference → both TSVs → `validate_submission.py --check-ids` must PASS → submission zip.
- Upgrades: French legal-form suffixes + dotted variants, abbreviation-finder regexes, token signatures, `sparse_dot_topn` + Soundex backfill blocking, inference short-circuit, `model_config.json` artifact.
- Anti-patterns to avoid: sklearn-macro thresholding (wrong metric), 1-to-1 bipartite matching (GT is multi-match), `.toarray()` densification, pure-Python Levenshtein on hot paths.


In [ ]:
import json
print(json.dumps({"mode": MODE, "sample_S1": len(sample_s1), "pool_rows": len(pool_df),
                   "outputs": sorted(str(p) for p in OUT_DIR.glob("*.tsv")),
                   "rss_gb": round(rss_gb(), 2)}, indent=2))